# Testing csprng on CW

## Connecting to ChipWhisperer

Now that your hardware is all setup, we can now learn how to connect to it. We can connect to the ChipWhisperer with:

In [1]:
import chipwhisperer as cw
scope = cw.scope()

In [2]:
target = cw.target(scope) #cw.targets.SimpleSerial can be omitted

We'll only be discussing the default target type, which is SimpleSerial. Other targets, like the CW305, will be covered in hardware specific demos. 

Some sane default settings can be set using:

In [3]:
scope.default_setup()

scope.gain.mode                          changed from low                       to high                     
scope.gain.gain                          changed from 0                         to 30                       
scope.gain.db                            changed from 5.5                       to 24.8359375               
scope.adc.basic_mode                     changed from low                       to rising_edge              
scope.adc.samples                        changed from 24400                     to 5000                     
scope.adc.trig_count                     changed from 247920583                 to 271566187                
scope.clock.adc_src                      changed from clkgen_x1                 to clkgen_x4                
scope.clock.adc_freq                     changed from 96000000                  to 29538459                 
scope.clock.adc_rate                     changed from 96000000.0                to 29538459.0               
scope.clock.clkgen_

which from its [documentation](https://chipwhisperer.readthedocs.io/en/latest/scope-api.html#chipwhisperer.scopes.OpenADC.default_setup) you can see does the following for the CWLite/CW1200:

* Sets the scope gain to 45dB
* Sets the scope to capture 5000 samples
* Sets the scope offset to 0 (aka it will begin capturing as soon as it is triggered)
* Sets the scope trigger to rising edge
* Outputs a 7.37MHz clock to the target on HS2
* Clocks the scope ADC at 4\*7.37MHz. Note that this is *synchronous* to the target clock on HS2
* Assigns GPIO1 as serial RX
* Assigns GPIO2 as serial TX

And that's it! Your ChipWhisperer is now setup and ready to attack a target. 

**NOTE: You'll need to disconnect the scope/target before connecting again, like you would in another notebook. This can be done with `scope.dis()` and `target.dis()`**

## Building and Uploading Firmware

The next step in attacking a target is to get some firmware built and uploaded onto it. Most firmware for most ChipWhisperer targets can be built using our build system, provided you have the correct compiler installed (see https://chipwhisperer.readthedocs.io/en/latest/installation.html for info about compilers).

Firmware must be built on the command line. Luckily, thanks to Jupyter, we can run a command within a notebook as follows:

In [6]:
%%bash
cd ../firmware/mcu/csprng-test/
make PLATFORM=CWLITEARM CRYPTO_TARGET=NONE

SS_VER set to SS_VER_1_1
SS_VER set to SS_VER_1_1
.
Welcome to another exciting ChipWhisperer target build!!
arm-none-eabi-gcc (15:13.2.rel1-2) 13.2.1 20231009
Copyright (C) 2023 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.

mkdir -p objdir-CWLITEARM 
.
Compiling:
-en     csprng.c ...


In file included from csprng.c:25:
csprng_hash.h: In function 'csprng_fp_mat_trigger':
csprng_hash.h:135:5: warning: implicit declaration of function 'trigger_high' [-Wimplicit-function-declaration]
  135 |     trigger_high();
      |     ^~~~~~~~~~~~
csprng_hash.h:139:5: warning: implicit declaration of function 'trigger_low' [-Wimplicit-function-declaration]
  139 |     trigger_low();
      |     ^~~~~~~~~~~
csprng.c: In function 'main':
csprng.c:238:5: warning: implicit declaration of function 'platform_init' [-Wimplicit-function-declaration]
  238 |     platform_init();
      |     ^~~~~~~~~~~~~
csprng.c:239:9: warning: implicit declaration of function 'init_uart' [-Wimplicit-function-declaration]
  239 |         init_uart();
      |         ^~~~~~~~~
csprng.c:240:9: warning: implicit declaration of function 'trigger_setup' [-Wimplicit-function-declaration]
  240 |         trigger_setup();
      |         ^~~~~~~~~~~~~


-e Done!
.
Compiling:
-en     csprng_hash.c ...


In file included from csprng_hash.c:34:
csprng_hash.h: In function 'csprng_fp_mat_trigger':
csprng_hash.h:135:5: warning: implicit declaration of function 'trigger_high' [-Wimplicit-function-declaration]
  135 |     trigger_high();
      |     ^~~~~~~~~~~~
csprng_hash.h:139:5: warning: implicit declaration of function 'trigger_low' [-Wimplicit-function-declaration]
  139 |     trigger_low();
      |     ^~~~~~~~~~~


-e Done!
.
Compiling:
-en     fips202.c ...
-e Done!
.
Compiling:
-en     keccakf1600.c ...
-e Done!
.
Compiling:
-en     randombytes.c ...


randombytes.c:38:2: warning: #warning Using a non-random randombytes [-Wcpp]
   38 | #warning Using a non-random randombytes
      |  ^~~~~~~


-e Done!
.
Compiling:
-en     .././simpleserial/simpleserial.c ...


.././simpleserial/simpleserial.c: In function 'simpleserial_get':
.././simpleserial/simpleserial.c:371:13: warning: implicit declaration of function 'getch' [-Wimplicit-function-declaration]
  371 |         c = getch();
      |             ^~~~~
.././simpleserial/simpleserial.c: In function 'simpleserial_put':
.././simpleserial/simpleserial.c:432:9: warning: implicit declaration of function 'putch' [-Wimplicit-function-declaration]
  432 |         putch(c);
      |         ^~~~~


-e Done!
.
Compiling:
-en     .././hal/hal.c ...
-e Done!
.
Compiling:
-en     .././hal//stm32f3/stm32f3_hal.c ...
-e Done!
.
Compiling:
-en     .././hal//stm32f3/stm32f3_hal_lowlevel.c ...
-e Done!
.
Compiling:
-en     .././hal//stm32f3/stm32f3_sysmem.c ...
-e Done!
.
Assembling: .././hal//stm32f3/stm32f3_startup.S
arm-none-eabi-gcc -c -mcpu=cortex-m4 -I. -x assembler-with-cpp -mthumb -mfloat-abi=soft -fmessage-length=0 -ffunction-sections -DF_CPU=7372800 -Wa,-gstabs,-adhlns=objdir-CWLITEARM/stm32f3_startup.lst -I.././simpleserial/ -I.././hal/ -I.././hal/ -I.././hal//stm32f3 -I.././hal//stm32f3/CMSIS -I.././hal//stm32f3/CMSIS/core -I.././hal//stm32f3/CMSIS/device -I.././hal//stm32f4/Legacy -I.././simpleserial/ -I.././crypto/ .././hal//stm32f3/stm32f3_startup.S -o objdir-CWLITEARM/stm32f3_startup.o
.
LINKING:
-en     csprng-CWLITEARM.elf ...
Memory region         Used Size  Region Size  %age Used
             RAM:        5600 B        40 KB     13.67%
             ROM:       13640 B   

You should see a big list of `PLATFORM`s to build for. We left the `PLATFORM` blank in the command above, so the build system instead printed a list of supported platforms. Fill in your platform, rerun the build command, and your firmware should be successfully built.

Continuing on, there's two possible ways to upload firmware to your target:

1. ChipWhisperer has built in support for XMEGA, STM32F*, AVR, and SAM4S bootloaders. These can be used as follows

In [9]:
#cw.program_target(scope, cw.programmers.XMEGAProgrammer, "path/to/firmware.hex")
cw.program_target(scope, cw.programmers.STM32FProgrammer, "../firmware/mcu/csprng-test/csprng-CWLITEARM.hex")
#cw.program_target(scope, cw.programmers.AVRProgrammer, "path/to/firmware.hex")
#cw.program_target(scope, cw.programmers.SAM4SProgrammer, "path/to/firmware.hex")

Detected known STMF32: STM32F302xB(C)/303xB(C)
Extended erase (0x44), this can take ten seconds or more
Attempting to program 13639 bytes at 0x8000000
STM32F Programming flash...
STM32F Reading flash...
Verified flash OK, 13639 bytes


2. For other targets, you'll need to use an external programmer or a debugger to flash the firmware onto the target. 

Whatever your case, upload the firmware we built earlier to the target device. Next we'll be learning how to capture power traces and communicate with the target.

## Basic test

In [10]:
msg = bytearray([0]*16) #simpleserial uses bytearrays
target.simpleserial_write('p', msg)

Let's check if we got a response:

In [11]:
print(target.simpleserial_read('r', 16))

bytearray(b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00')


### Generate the matrix
This command sends no response

In [12]:
msg = bytearray([0,0]) #simpleserial uses bytearrays
target.simpleserial_write('m', msg)

In [13]:
diff = target.simpleserial_read('r', 2)

In [14]:
diff_num = int.from_bytes(diff, byteorder='big', signed=False)
print(diff_num)

0


### Read the matrix
The command 'g' should send 16 of the matrix's values

In [15]:
n = 127
k = 76
rounds = (n-k)*k//38

In [16]:
def matrix_of(data):
    res = [[0x00 for _ in range(n-k)] for _ in range(k)]
    print((n-k)*k, len(data), rounds)
    for i in range(k) :
        for j in range(n-k) :
            if i*(n-k)+j < (n-k)*k :
                res[i][j]= data[i*(n-k)+j]
    return res

def pp_matrix(matrix) :
    s = [[str(e) for e in row] for row in matrix]
    lens = [max(map(len, col)) for col in zip(*s)]
    fmt = '\t'.join('{{:{}}}'.format(x) for x in lens)
    table = [fmt.format(*row) for row in s]
    print('\n'.join(table))

In [17]:
msg = bytearray([0]*38)
data = bytearray()
for i in range(rounds) :
    # print("["+hex(i*16)+"]")
    target.simpleserial_write('g', msg)
    buff = target.simpleserial_read('r', 38)
    data.extend(buff)
    # print(buff)


In [18]:
print(len(data))
mat = matrix_of(data)
pp_matrix(mat)

3876
3876 3876 102
72 	3  	95 	79 	126	11 	81 	4  	100	40 	78 	1  	66 	67 	99 	89 	76 	44 	25 	3  	117	110	123	121	70 	111	88 	18 	95 	98 	37 	53 	85 	45 	86 	114	54 	71 	121	96 	83 	97 	28 	4  	29 	108	54 	7  	24 	102	31 
105	115	21 	41 	119	113	60 	0  	122	38 	2  	113	109	124	117	20 	71 	76 	46 	68 	35 	95 	104	96 	10 	82 	41 	102	113	101	20 	43 	86 	36 	19 	98 	96 	89 	76 	126	39 	102	17 	81 	72 	102	72 	101	49 	92 	56 
74 	28 	2  	31 	110	20 	36 	16 	82 	61 	112	92 	84 	10 	76 	86 	122	40 	71 	78 	78 	121	99 	43 	7  	121	29 	41 	23 	72 	74 	29 	40 	98 	36 	104	104	66 	126	48 	29 	41 	30 	52 	106	94 	54 	16 	117	1  	24 
41 	104	6  	117	17 	93 	18 	33 	4  	30 	92 	53 	66 	48 	107	94 	41 	26 	6  	18 	0  	64 	75 	28 	57 	119	96 	13 	110	51 	113	58 	21 	113	84 	81 	59 	69 	15 	53 	28 	7  	65 	56 	55 	125	67 	3  	4  	64 	96 
12 	73 	48 	23 	69 	49 	76 	87 	68 	100	24 	123	57 	91 	31 	80 	14 	69 	111	49 	73 	54 	45 	34 	35 	64 	40 	0  	24 	107	46 	124	103	105	86 	68 	121	100	48 	101	40 	3

In [19]:
target.flush()
msg = bytearray()
target.simpleserial_write('r', msg)

## Disconnecting

Disconnect from the hardware so it doesn't stay "in use" by this notebook : 

In [20]:
scope.dis()
target.dis()